# Recepcion

In [1]:
import sys 
sys.path.append('C:/Users/A365/Documents/script_01/resumen')
from funciones import *


In [2]:

from pyspark.sql import functions as F
# from pyspark.sql.functions import last_day, col
from dateutil.relativedelta import relativedelta

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-12.8.1.jre8.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-12.8.1.jre8.jar') \
    .config('spark.executor.memory', '20g') \
    .config('spark.driver.memory', '20g') \
    .getOrCreate()

In [3]:
def add_base(db_campana,campaign_name,nombre_skill,fecha_a):
    query = f"""
    select 
        a.contact_info,
        a.chain_id,
        b.nombre_base_cargada,
        b.skill,
        b.nombre_skill,
        b.fecha_carga,
        b.n_sem,
        b.fecha_final,
        case
            when b.tipo_base is null then 3
            when b.tipo_base='BASE' then 1
            else 2
        end as tipo_base
    from tb_RegistroBase a
    inner join DB_BaseSemanal.dbo.tb_listaBaseCargada b
    on a.id_baseCargada=b.id_baseCargada
    where b.skill='{campaign_name}'
    and b.nombre_skill='{nombre_skill}'
    AND B.fecha_inicio between DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_a}')-1 , 0) and EOMONTH('{fecha_a}')
    """
    return obtener_tabla_sql(spark, db_campana, query)

def add_base_1(db_campana,campaign_name,nombre_skill,fecha_a):
    query = f"""
    select 
        a.contact_info,
        a.chain_id as chain_id1,
        b.nombre_base_cargada as nombre_base_cargada1,
        b.nombre_skill as nombre_skill1,
        b.fecha_carga as fecha_carga1,
        case
            when b.tipo_base is null then 3
            when b.tipo_base='BASE' then 1
            else 2
        end as tipo_base1
    from tb_RegistroBase a
    inner join DB_BaseSemanal.dbo.tb_listaBaseCargada b
    on a.id_baseCargada=b.id_baseCargada
    where b.skill='{campaign_name}'
    and b.nombre_skill='{nombre_skill}'
    AND B.fecha_inicio >= DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_a}')-1 , 0) 
    """
    return obtener_tabla_sql(spark, db_campana, query)

fecha_a='2025-02-01'
df_recu_base=add_base('DB_RECUPERADOS','CMP_CL_OUT_ECO_A365_PORTA_RECUPERADOS','PORTABILIDAD_RECUPERADOS',fecha_a)
df_recu_base = df_recu_base.withColumn("indice", row_number().over(Window.orderBy(monotonically_increasing_id())))

fecha_a='2024-11-01'
df_01_base=add_base_1('DB_MIGRACIONES','CMP_CL_OUT_ECO_A365_MIGRACIONES','MIGRACIONES_1',fecha_a)
df_02_base=add_base_1('DB_MIGRACIONES','CMP_CL_OUT_ECO_A365_MIGRACIONES2','MIGRACIONES_2',fecha_a)
df_03_base=add_base_1('DB_IVR','CMP_CL_OUT_ECO_A365_PORTA_IVR','PORTABILIDAD_PREPAGO_IVR',fecha_a)
df_04_base=add_base_1('DB_PERFILADA','CMP_CL_OUT_ECO_A365_PORTA_PERFILADA','PORTABILIDAD_PERFILADA_1',fecha_a)
df_05_base=add_base_1('DB_RECUPERADOS','CMP_CL_OUT_ECO_A365_PORTA_RECUPERADOS','PORTABILIDAD_RECUPERADOS',fecha_a)
df_06_base=add_base_1('DB_SEGUNDAS','CMP_CL_OUT_ECO_A365_SEGUNDAS_LINEAS','SEGUNDAS_LINEAS',fecha_a)
df_tot=df_01_base.unionByName(df_01_base).unionByName(df_02_base).unionByName(df_03_base).unionByName(df_04_base).unionByName(df_05_base).unionByName(df_06_base)


+----------+
|     fecha|
+----------+
|2025-02-01|
|2025-02-02|
|2025-02-03|
|2025-02-04|
|2025-02-05|
+----------+



In [4]:
df_recu_base=df_recu_base.join(df_tot,['contact_info'],'left')

df_recu_base = df_recu_base.withColumn(
    "tipo_base1",
    F.when(F.col("tipo_base1").isNull(), 0)
    .when(F.col("nombre_base_cargada") == F.col("nombre_base_cargada1"), 0)
    .otherwise(F.col("tipo_base1"))
)
df_recu_base = df_recu_base.withColumn(
    "q_descanso",
    F.when(F.col("fecha_carga") == F.col("fecha_carga1"), 0)
    .otherwise(F.datediff(F.col("fecha_carga"), F.col("fecha_carga1")))
)
df_recu_base = df_recu_base.withColumn(
    "q_descanso",
    F.when(F.col("fecha_carga1").isNull(), 0)
    .otherwise(0)
)

In [5]:
df_recu_base = df_recu_base.select(
    "*", 
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'MIGRACIONES_1'), F.col("fecha_carga1")).alias("fecha_m1"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'MIGRACIONES_1'), F.col("q_descanso")).otherwise(0).alias("descanso_m1"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'MIGRACIONES_1'), F.col("nombre_base_cargada1")).otherwise('sin resultado').alias("nombre_base_cargada_m1"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'MIGRACIONES_1'), 1).otherwise(0).alias("unico_m1"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'MIGRACIONES_1'), 1).otherwise(0).alias("tipo_base1_m1"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'MIGRACIONES_1'), F.col("chain_id1")).otherwise(0).alias("chain_id_m1"),
     
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'MIGRACIONES_2'), F.col("fecha_carga1")).alias("fecha_m2"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'MIGRACIONES_2'), F.col("q_descanso")).otherwise(0).alias("descanso_m2"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'MIGRACIONES_2'), F.col("nombre_base_cargada1")).otherwise('sin resultado').alias("nombre_base_cargada_m2"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'MIGRACIONES_2'), 1).otherwise(0).alias("unico_m2"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'MIGRACIONES_2'), 1).otherwise(0).alias("tipo_base1_m2"), 
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'MIGRACIONES_2'), F.col("chain_id1")).otherwise(0).alias("chain_id_m2"),

    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'SEGUNDAS_LINEAS'), F.col("fecha_carga1")).alias("fecha_seg"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'SEGUNDAS_LINEAS'), F.col("q_descanso")).otherwise(0).alias("descanso_seg"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'SEGUNDAS_LINEAS'), F.col("nombre_base_cargada1")).otherwise('sin resultado').alias("nombre_base_cargada_seg"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'SEGUNDAS_LINEAS'), 1).otherwise(0).alias("unico_seg"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'SEGUNDAS_LINEAS'), 1).otherwise(0).alias("tipo_base1_seg"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'SEGUNDAS_LINEAS'), F.col("chain_id1")).otherwise(0).alias("chain_id_seg"),

    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_PERFILADA_1'), F.col("fecha_carga1")).alias("fecha_per1"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_PERFILADA_1'), F.col("q_descanso")).otherwise(0).alias("descanso_per1"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_PERFILADA_1'), F.col("nombre_base_cargada1")).otherwise('sin resultado').alias("nombre_base_cargada_per1"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_PERFILADA_1'), 1).otherwise(0).alias("unico_per1"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_PERFILADA_1'), 1).otherwise(0).alias("tipo_base1_per1"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_PERFILADA_1'), F.col("chain_id1")).otherwise(0).alias("chain_id_per1"),

    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_PERFILADA_2'), F.col("fecha_carga1")).alias("fechaper2"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_PERFILADA_2'), F.col("q_descanso")).otherwise(0).alias("descanso_per2"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_PERFILADA_2'), F.col("nombre_base_cargada1")).otherwise('sin resultado').alias("nombre_base_cargadaper2"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_PERFILADA_2'), 1).otherwise(0).alias("unico_per2"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_PERFILADA_2'), 1).otherwise(0).alias("tipo_base1per2"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_PERFILADA_2'), F.col("chain_id1")).otherwise(0).alias("chain_id_per2"),

    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_RECUPERADOS'), F.col("fecha_carga1")).alias("fecha_recu"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_RECUPERADOS'), F.col("q_descanso")).otherwise(0).alias("descanso_recu"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_RECUPERADOS'), F.col("nombre_base_cargada1")).otherwise('sin resultado').alias("nombre_base_cargada_recu"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_RECUPERADOS'), 1).otherwise(0).alias("unico_recu"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_RECUPERADOS'), 1).otherwise(0).alias("tipo_base1_recu"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_RECUPERADOS'), F.col("chain_id1")).otherwise(0).alias("chain_id_recu"),

    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_PREPAGO_IVR'), F.col("fecha_carga1")).alias("fecha_ivr"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_PREPAGO_IVR'), F.col("q_descanso")).otherwise(0).alias("descanso_ivr"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_PREPAGO_IVR'), F.col("nombre_base_cargada1")).otherwise('sin resultado').alias("nombre_base_cargada_ivr"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_PREPAGO_IVR'), 1).otherwise(0).alias("unico_ivr"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_PREPAGO_IVR'), 1).otherwise(0).alias("tipo_base1_ivr"),
    F.when((F.col("fecha_carga1").isNotNull()) & (F.col("nombre_skill1") == 'PORTABILIDAD_PREPAGO_IVR'), F.col("chain_id1")).otherwise(0).alias("chain_id_ivr")
)


In [ ]:
['contact_info', 'chain_id', 'nombre_base_cargada', 'skill', 'nombre_skill', 'fecha_carga', 'n_sem', 'fecha_final', 'tipo_base', 'indice', 'chain_id1', 'nombre_base_cargada1', 'nombre_skill1', 'fecha_carga1', 'tipo_base1', 'q_descanso', 'fecha_m1', 'descanso_m1', 'nombre_base_cargada_m1', 'descanso_m1', 'tipo_base1_m1', 'chain_id_m1', 'fecha_m2', 'descanso_m2', 'nombre_base_cargada_m2', 'descanso_m2', 'tipo_base1_m2', 'chain_id_m2', 'fecha_seg', 'descanso_seg', 'nombre_base_cargada_seg', 'descanso_seg', 'tipo_base1_seg', 'chain_id_seg', 'fecha_per1', 'descanso_per1', 'nombre_base_cargada_per1', 'descanso_per1', 'tipo_base1_per1', 'chain_id_per1', 'fechaper2', 'descansoper2', 'nombre_base_cargadaper2', 'descansoper2', 'tipo_base1per2', 'chain_id_per2', 'fecha_recu', 'descanso_recu', 'nombre_base_cargada_recu', 'descanso_recu', 'tipo_base1_recu', 'chain_id_recu', 'fecha_ivr', 'descanso_ivr', 'nombre_base_cargada_ivr', 'descanso_ivr', 'tipo_base1_ivr', 'chain_id_ivr']

['contact_info', 'chain_id', 'nombre_base_cargada', 'skill', 'nombre_skill', 'fecha_carga', 'n_sem', 'fecha_final', 'tipo_base', 'indice', 'chain_id1', 'nombre_base_cargada1', 'nombre_skill1', 'fecha_carga1', 'tipo_base1', 'q_descanso', 'fecha_m1', 'descanso_m1', 'nombre_base_cargada_m1', 'descanso_m1', 'tipo_base1_m1', 'chain_id_m1', 'fecha_m2', 'descanso_m2', 'nombre_base_cargada_m2', 'descanso_m2', 'tipo_base1_m2', 'chain_id_m2', 'fecha_seg', 'descanso_seg', 'nombre_base_cargada_seg', 'descanso_seg', 'tipo_base1_seg', 'chain_id_seg', 'fecha_per1', 'descanso_per1', 'nombre_base_cargada_per1', 'descanso_per1', 'tipo_base1_per1', 'chain_id_per1', 'fechaper2', 'descansoper2', 'nombre_base_cargadaper2', 'descansoper2', 'tipo_base1per2', 'chain_id_per2', 'fecha_recu', 'descanso_recu', 'nombre_base_cargada_recu', 'descanso_recu', 'tipo_base1_recu', 'chain_id_recu', 'fecha_ivr', 'descanso_ivr', 'nombre_base_cargada_ivr', 'descanso_ivr', 'tipo_base1_ivr', 'chain_id_ivr']


In [6]:

window_spec = Window.partitionBy('indice').orderBy(col("fecha_carga").asc())
df_recu_base = df_recu_base.withColumn("unico", row_number().over(window_spec))

df_recu_base = df_recu_base.withColumn(
    "unico",
    F.when(F.col("unico")==1, 1)
    .otherwise(0)
)

In [7]:

window_spec = Window.partitionBy('nombre_base_cargada','nombre_skill1','contact_info').orderBy(col("fecha_carga").asc())
df_recu_base = df_recu_base.withColumn("unico", row_number().over(window_spec))


df_ccr_campana= df_ccr_campana.filter(col("unico_1") == 1).drop('unico_1','indice','resource_name')

NameError: name 'df_ccr_campana' is not defined

In [8]:
overwrite_table_SQL(df_recu_base,'DB_temporal','recepcion_recuperados_febrero','')